In [ ]:
# =============================
# Stitching optical+laser images together
# =============================
import cv2
import numpy as np
from PIL import Image
from openpyxl import load_workbook


def stitch_by_position(xlsx_path, grid_size=(13, 13)):
    rows, cols = grid_size

    wb = load_workbook(xlsx_path, data_only=True)
    ws = wb.active

    # Sort images by their position (top → bottom, left → right)
    def get_pos(img):
        try:
            return (img.anchor._from.row, img.anchor._from.col)
        except:
            return (9999, 9999)

    images = sorted(ws._images, key=get_pos)

    print(f"Using {len(images)} images")

    tiles = {}

    i = 0
    for y in range(1, rows + 1):
        for x in range(1, cols + 1):
            if i >= len(images):
                break

            img_obj = images[i]
            i += 1

            try:
                img_obj.ref.seek(0)
                pil_img = Image.open(img_obj.ref).convert("RGB")
                image = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
            except Exception as e:
                print(f"Failed to read image {i}: {e}")
                continue

            tiles[(y, x)] = image

    if not tiles:
        raise ValueError("No images loaded")

    # Normalize size
    sample = next(iter(tiles.values()))
    h, w = sample.shape[:2]

    stitched = np.zeros((rows * h, cols * w, 3), dtype=np.uint8)

    for (y, x), img in tiles.items():
        if img.shape[:2] != (h, w):
            img = cv2.resize(img, (w, h))

        y0 = (y - 1) * h
        x0 = (x - 1) * w
        stitched[y0:y0+h, x0:x0+w] = img

    print(f"Placed {len(tiles)}/{rows*cols} tiles")

    return stitched


# RUN
xlsx_path = "E1ImageStitching.xlsx"

result = stitch_by_position(xlsx_path, (13, 13))
cv2.imwrite("stitched.png", result)

print("Saved stitched.png")